In [26]:
#https://www.codenong.com/20186344/

%run BaseInfo_Company.ipynb
%run BaseInfo_ETFNo.ipynb

In [27]:
import pandas as pd

In [28]:
StockBaseData=pd.read_csv('StockBaseData.csv')
ETFBaseData=pd.read_csv('ETFBaseData.csv')

In [29]:
StockBaseData.head(2)

,公司代號,公司名稱,公司簡稱,產業別,上市公司產業類別,上櫃公司產業類別,新編碼
0,1101,台灣水泥股份有限公司,台泥,1,水泥工業,--,1.0
1,1102,亞洲水泥股份有限公司,亞泥,1,水泥工業,--,1.0


In [30]:
ETFBaseData.head(2)

,股票名稱,持股(千股),比例,增減,ETF_Id
0,台積電,"151,043.00",47.11,-1.83%,50
1,聯發科,"9,300.00",4.69,-0.34%,50


## 資料處理



In [31]:
# 兩表Join
df=pd.merge(ETFBaseData, StockBaseData, how='left', left_on='股票名稱', right_on='公司簡稱')
# 處理格式問題
df['ETF_Id']='00'+df['ETF_Id'].astype(str)

#處理上市公司產業類別對應不到的狀況
df['上市公司產業類別']=df['上市公司產業類別'].fillna('未分類')
print(df.columns)


Index(['股票名稱', '持股(千股)', '比例', '增減', 'ETF_Id', '公司代號', '公司名稱', '公司簡稱', '產業別',
       '上市公司產業類別', '上櫃公司產業類別', '新編碼'],
      dtype='object')


In [32]:
df[df['ETF_Id']=='00891']

,股票名稱,持股(千股),比例,增減,ETF_Id,公司代號,公司名稱,公司簡稱,產業別,上市公司產業類別,上櫃公司產業類別,新編碼
1424,聯發科,"1,912.00",19.13,-0.54%,00891,2454.0,聯發科技股份有限公司,聯發科,24.0,半導體業,半導體業,24.0
1425,台積電,"3,057.00",18.92,+0.05%,00891,2330.0,台灣積體電路製造股份有限公司,台積電,24.0,半導體業,半導體業,24.0
1426,聯電,"19,642.00",10.85,+0.09%,00891,2303.0,聯華電子股份有限公司,聯電,24.0,半導體業,半導體業,24.0
1427,日月光投控,"5,416.00",6.31,+0.07%,00891,3711.0,日月光投資控股股份有限公司,日月光投控,24.0,半導體業,半導體業,24.0
1428,矽力-KY,138.00,5.44,+0.20%,00891,6415.0,矽力杰股份有限公司,矽力-KY,24.0,半導體業,半導體業,24.0
1429,聯詠,"1,013.00",5.26,-0.34%,00891,3034.0,聯詠科技股份有限公司,聯詠,24.0,半導體業,半導體業,24.0
1430,瑞昱,790.00,4.15,+0.08%,00891,2379.0,瑞昱半導體股份有限公司,瑞昱,24.0,半導體業,半導體業,24.0
1431,環球晶,365.00,3.49,+0.38%,00891,NaN,NaN,NaN,NaN,未分類,NaN,NaN
1432,穩懋,569.00,2.22,+0.34%,00891,NaN,NaN,NaN,NaN,未分類,NaN,NaN
1433,世界,"1,564.00",1.92,+0.05%,00891,NaN,NaN,NaN,NaN,未分類,NaN,NaN


In [33]:
# 取得 etf 對應持股佔比
df['data']=df.agg(lambda x: f"{x['股票名稱']} ( {x['比例']}%)", axis=1)
df.sort_values('比例', ascending=False)
t=df[['ETF_Id','data']]
t[t['ETF_Id']=='00891']

,ETF_Id,data
1424,00891,聯發科 ( 19.13%)
1425,00891,台積電 ( 18.92%)
1426,00891,聯電 ( 10.85%)
1427,00891,日月光投控 ( 6.31%)
1428,00891,矽力-KY ( 5.44%)
1429,00891,聯詠 ( 5.26%)
1430,00891,瑞昱 ( 4.15%)
1431,00891,環球晶 ( 3.49%)
1432,00891,穩懋 ( 2.22%)
1433,00891,世界 ( 1.92%)


In [34]:
keydata=df.groupby(['ETF_Id','上市公司產業類別'])['data'].apply(','.join).reset_index()
keydata

,ETF_Id,上市公司產業類別,data
0,0050,光電業,"大立光 ( 1.05%),友達 ( 0.66%)"
1,0050,其他,"中租-KY ( 0.84%),豐泰 ( 0.36%)"
2,0050,其他電子業,鴻海 ( 4.41%)
3,0050,半導體業,"台積電 ( 47.11%),聯發科 ( 4.69%),聯電 ( 2.04%),日月光投控 (..."
4,0050,塑膠工業,"台塑 ( 1.65%),南亞 ( 1.55%),台化 ( 0.96%)"
...,...,...,...
355,00891,電子零組件業,健策 ( 0.34%)
356,00892,半導體業,"台積電 ( 23.73%),瑞昱 ( 5.91%),聯發科 ( 5.76%),聯電 ( 5...."
357,00892,未分類,"環球晶 ( 6.73%),穩懋 ( 5.72%),世界 ( 4.89%),譜瑞-KY ( 4..."
358,00892,通信網路業,全新 ( 0.98%)


In [35]:
# 取得產業比例排名 by ETF_Id 上市公司產業類別

# 挑選要用的欄位
colList=['ETF_Id','股票名稱', '上市公司產業類別','持股(千股)','比例']
df_=df[colList]
# groupby
g=df_.groupby(['ETF_Id','上市公司產業類別']).sum().reset_index()
# 排序 取前三
#g.sort_values('比例', ascending=False).groupby(['ETF_Id']).head(3).sort_values(['ETF_Id', '比例'], ascending=[True, False])
# 排序(這個對未來比較有彈性)
g['row_numbers'] =g.groupby('ETF_Id')['比例'].rank(ascending=False,method='dense')
g[g['ETF_Id']=='00891']

,ETF_Id,上市公司產業類別,比例,row_numbers
352,00891,半導體業,80.67,1.0
353,00891,未分類,14.98,2.0
354,00891,電子通路業,1.69,3.0
355,00891,電子零組件業,0.34,4.0


In [36]:
#轉成橫的方便看
df1=g[g['row_numbers']==1].merge(keydata,on=['ETF_Id','上市公司產業類別'])
df2=g[g['row_numbers']==2].merge(keydata,on=['ETF_Id','上市公司產業類別'])
df3=g[g['row_numbers']==3].merge(keydata,on=['ETF_Id','上市公司產業類別'])
df_result=df1.merge(df2,on='ETF_Id',suffixes=["","_2"]).merge(df3,on='ETF_Id',suffixes=["","_3"])
df_result = df_result.drop(df_result.filter(regex='row_numbers').columns, axis=1)
df_result

,ETF_Id,上市公司產業類別,比例,data,上市公司產業類別_2,比例_2,data_2,上市公司產業類別_3,比例_3,data_3
0,0050,半導體業,57.81,"台積電 ( 47.11%),聯發科 ( 4.69%),聯電 ( 2.04%),日月光投控 (...",金融保險,11.24,"富邦金 ( 1.79%),國泰金 ( 1.47%),中信金 ( 1.36%),兆豐金 ( 1...",其他電子業,4.41,鴻海 ( 4.41%)
1,0051,電子零組件業,13.08,"欣興 ( 2.61%),華新科 ( 1.75%),健鼎 ( 1.07%),臻鼎-KY ( 1...",電腦及週邊設備業,12.22,"光寶科 ( 1.93%),微星 ( 1.71%),仁寶 ( 1.45%),緯創 ( 1.37...",金融保險,10.75,"開發金 ( 2.9%),永豐金 ( 2.29%),新光金 ( 1.79%),中壽 ( 0.8..."
2,0052,半導體業,75.34,"台積電 ( 59.26%),聯發科 ( 5.95%),聯電 ( 2.59%),日月光投控 (...",電子零組件業,7.40,"台達電 ( 2.74%),國巨 ( 1.06%),欣興 ( 0.69%),華新科 ( 0.4...",其他電子業,6.73,"鴻海 ( 5.59%),可成 ( 0.56%),鴻準 ( 0.3%),旭隼 ( 0.28%)"
3,0053,半導體業,59.31,"台積電 ( 43.35%),聯發科 ( 4.03%),聯電 ( 1.93%),日月光投控 (...",電子零組件業,8.60,"台達電 ( 2.22%),國巨 ( 0.78%),南電 ( 0.71%),欣興 ( 0.52...",電腦及週邊設備業,6.89,"廣達 ( 0.97%),華碩 ( 0.8%),研華 ( 0.74%),緯穎 ( 0.54%)..."
4,0054,其他電子業,20.09,"鴻海 ( 14.65%),可成 ( 1.78%),旭隼 ( 1.42%),致茂 ( 1.01...",電子零組件業,17.56,"台達電 ( 7.63%),國巨 ( 2.8%),華新科 ( 1.03%),台光電 ( 0.8...",塑膠工業,10.62,"南亞 ( 6.13%),台化 ( 4.49%)"
5,0056,電腦及週邊設備業,27.84,"緯創 ( 3.88%),仁寶 ( 3.29%),華碩 ( 3.27%),廣達 ( 3.03%...",光電業,11.58,"友達 ( 4.74%),群創 ( 3.82%),瑞儀 ( 3.02%)",航運業,8.91,長榮 ( 8.91%)
6,0057,半導體業,53.42,"台積電 ( 42.95%),聯發科 ( 4.25%),聯電 ( 1.83%),日月光投控 (...",金融保險,10.98,"富邦金 ( 1.44%),國泰金 ( 1.25%),中信金 ( 1.23%),兆豐金 ( 1...",其他電子業,4.61,"鴻海 ( 4.08%),可成 ( 0.35%),鴻準 ( 0.18%)"
7,006203,半導體業,53.92,"台積電 ( 43.26%),聯發科 ( 4.27%),聯電 ( 1.85%),日月光投控 (...",金融保險,11.05,"富邦金 ( 1.45%),國泰金 ( 1.26%),中信金 ( 1.24%),兆豐金 ( 1...",其他電子業,4.68,"鴻海 ( 4.12%),可成 ( 0.38%),鴻準 ( 0.18%)"
8,006204,半導體業,32.23,"台積電 ( 24.03%),聯發科 ( 2.0%),聯電 ( 1.12%),日月光投控 ( ...",航運業,9.01,"陽明 ( 4.06%),長榮 ( 2.65%),萬海 ( 1.17%),裕民 ( 0.42%...",金融保險,8.96,"國泰金 ( 1.04%),富邦金 ( 1.01%),元大金 ( 0.62%),兆豐金 ( 0..."
9,006208,半導體業,58.00,"台積電 ( 47.28%),聯發科 ( 4.7%),聯電 ( 2.05%),日月光投控 ( ...",金融保險,11.26,"富邦金 ( 1.8%),國泰金 ( 1.47%),中信金 ( 1.36%),兆豐金 ( 1....",其他電子業,4.42,鴻海 ( 4.42%)
